In [1]:
import pandas as pd
import re
from copy import deepcopy
from openpyxl import load_workbook
import numpy as np

Парсеры

In [2]:
def parse_specs(specs_part):
    """
    Парсит строку с dil спецификациями.
    
    Вход: 'dil 20 B1:B3, dil 10 C1:C3'
    Выход: [
        {'dilution': 20, 'wells': ['B1','B2','B3']},
        {'dilution': 10, 'wells': ['C1','C2','C3']}
    ]
    """
    specs = []
    
    # Разбиваем по 'dil' и пропускаем первый пустой элемент
    parts = specs_part.split('dil')[1:]
    
    for part in parts:
        part = part.strip().strip(',')
        if not part:
            continue
            
        tokens = part.split()
        dilution = int(tokens[0])
        wells_str = ' '.join(tokens[1:])
        
        wells = parse_wells(wells_str)
        
        specs.append({
            'dilution': dilution,
            'wells': wells
        })
    
    return specs

In [3]:
def parse_wells(parts):
    """'B1:B3, C5, E6:F6' -> ['B1','B2','B3','C5','E6','F6']"""
    wells = []
    COLUMNS = {
    'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7, 'H': 8, 'I':9, 'J':10, 'K':11, 'L':12}
    COLUMNS_REVERSED = {i: k for k, i in COLUMNS.items()}
    for part in parts:
        part = part.strip()
        if ':' in part:
            # Колонка или строчка
            start, end = part.split(':')
            start_letter = re.match(r'([A-Z]+)', start).group(1)
            start_num = int(re.match(r'[A-Z]+(\d+)', start).group(1))
            end_letter = re.match(r'([A-Z]+)', end).group(1)
            end_num = int(re.match(r'[A-Z]+(\d+)', end).group(1))
            
            if start_letter == end_letter:
                # Одна колонка
                for num in range(start_num, end_num + 1):
                    wells.append(f"{start_letter}{num}")
            elif start_num == end_num:
                #Cтрочка
                for i in range(COLUMNS[start_letter], COLUMNS[end_letter]+1):
                    wells.append(f"{COLUMNS_REVERSED[i]}{start_num}")
        else:
            # Одиночная ячейка
            wells.append(part)
    
    return wells


In [4]:
def parse_add(rest):
    # rest = '"образец 1" dil 20 B1:B3, dil 10 C1:C3'
    
    # 1. Имя
    name_match = re.search(r'"([^"]+)"', rest)
    name = name_match.group(1)
    
    # 2. Остаток после имени
    specs_part = rest.split('"')[2].strip()
    
    # 3. Парсим спецификации
    specs = parse_specs(specs_part)  # ← ЗДЕСЬ ВЫЗЫВАЕМ
    
    return {
        'action': 'add',
        'name': name,
        'specs': specs
    }

In [5]:
def parse_edit(rest):
    """
    Парсит команду edit.
    
    Вход: '"образец 1" B1:B3 dil 40'
    Выход: {
        'action': 'edit',
        'name': 'образец 1',
        'wells': ['B1','B2','B3'],
        'new_dilution': 40
    }
    """
    # 1. Имя в кавычках
    name_match = re.search(r'"([^"]+)"', rest)
    name = name_match.group(1)
    
    # 2. Всё что после имени
    after_name = rest.split('"')[2].strip()
    
    # 3. Разбиваем по пробелам
    tokens = after_name.split()
    
    # Ищем где стоит 'dil'
    dil_index = tokens.index('dil')
    
    # Всё до dil_index - лунки
    wells_str = ' '.join(tokens[:dil_index])
    # Всё после dil_index - число
    new_dilution = int(tokens[dil_index + 1])
    
    wells = parse_wells(wells_str)
    
    return {
        'action': 'edit',
        'name': name,
        'wells': wells,
        'new_dilution': new_dilution
    }

In [6]:
def parse_rename(rest):
    """
    Парсит команду rename.
    
    Вход: '"образец 1" "образец 2"'
    Выход: {
        'action': 'rename',
        'old_name': 'образец 1',
        'new_name': 'образец 2'
    }
    """
    # Находим все имена в кавычках
    matches = re.findall(r'"([^"]+)"', rest)
    
    if len(matches) != 2:
        raise ValueError(
            "rename требует два имени в кавычках: rename \"старое\" \"новое\"\n"
            f"Получено: {rest}"
        )
    
    return {
        'action': 'rename',
        'old_name': matches[0],
        'new_name': matches[1]
    }

In [7]:
def parse_delete(rest):
    """
    Парсит команду delete.
    
    Вход: '"образец 1"'
    Выход: {
        'action': 'delete',
        'name': 'образец 1'
    }
    """
    name_match = re.search(r'"([^"]+)"', rest)
    if not name_match:
        raise ValueError(
            "delete требует имя фрейма в кавычках: delete \"название\"\n"
            f"Получено: {rest}"
        )
    
    return {
        'action': 'delete',
        'name': name_match.group(1)
    }

In [8]:
def parse_show(rest):
    """
    Парсит команду show.
    
    Вход: '"образец 1"' или 'all'
    Выход: {
        'action': 'show',
        'target': 'frame',  # или 'all'
        'name': 'образец 1'  # только для frame
    }
    """
    rest = rest.strip()
    
    if rest == 'all':
        return {
            'action': 'show',
            'target': 'all'
        }
    
    # Если в кавычках - показываем фрейм
    name_match = re.search(r'"([^"]+)"', rest)
    if name_match:
        return {
            'action': 'show',
            'target': 'frame',
            'name': name_match.group(1)
        }
    
    raise ValueError(
        "show требует 'all' или имя фрейма в кавычках: show all / show \"название\"\n"
        f"Получено: {rest}"
    )

In [9]:
def parse_load(rest):
    """
    Парсит команду load.
    
    Вход: '"plate.xlsx" sheet "Sheet1"'
    Выход: {
        'action': 'load_full',
        'file_path': 'plate.xlsx',
        'sheet_name': 'Sheet1'
    }
    """
    # Ищем файл в кавычках
    file_match = re.search(r'"([^"]+)"', rest)
    if not file_match:
        raise ValueError(
            "load требует путь к файлу в кавычках: load \"file.xlsx\" sheet \"Лист1\"\n"
            f"Получено: {rest}"
        )
    file_path = file_match.group(1)
    
    # Ищем sheet в кавычках после sheet
    rest_after_file = rest.split('"')[2]
    sheet_match = re.search(r'sheet\s+"([^"]+)"', rest_after_file)
    if not sheet_match:
        raise ValueError(
            "load требует название листа: load \"file.xlsx\" sheet \"Лист1\"\n"
            f"Получено: {rest_after_file}"
        )
    sheet_name = sheet_match.group(1)
    
    return {
        'action': 'load_full',
        'file_path': file_path,
        'sheet_name': sheet_name
    }

In [10]:
def parse_command(text):
    """
    'add "образец 1" dil 20 B1:B3, dil 10 C1:C3'
    ↓
    {
        'action': 'add',
        'name': 'образец 1',
        'specs': [
            {'type': 'dil', 'value': 20, 'wells_str': 'B1:B3'},
            {'type': 'dil', 'value': 10, 'wells_str': 'C1:C3'}
        ]
    }
    """
    text = text.strip() #убирает пробелы слева от команды и справа от команды
    
    # 1. Откусывает первое слово-действие, деля по первому пробелу 
    parts = text.split(maxsplit=1)
    action = parts[0].lower()
    rest = parts[1] if len(parts) > 1 else '' #остаток
    
    # 2. Отправляем в соответствующий парсер
    if action == 'add':
        return parse_add(rest)
    elif action == 'edit':
        return parse_edit(rest)
    elif action == 'erase':
        return parse_erase(rest)
    elif action == 'rename':
        return parse_rename(rest)
    elif action == 'delete':
        return parse_delete(rest)
    elif action == 'show':
        return parse_show(rest)
    elif action == 'undo':
        return {'action': 'undo'}
    elif action == 'load': 
        return parse_load(rest)
    else:
        raise ValueError(f'Неизвестная команда: {action}')

Экзекьюторы

In [11]:
def execute_add(state, name, specs):
    """
    specs = [
        {'dilution': 20, 'wells': ['B1','B2','B3']},
        {'dilution': 10, 'wells': ['C1','C2','C3']}
    ]
    """
    new_state = deepcopy(state)
    
    # Создаем фрейм если нет
    if name not in new_state['frames']:
        new_state['frames'][name] = {}
    
    added = []
    conflicts = []
    
    for spec in specs:
        dilution = spec['dilution']
        wells = spec['wells']
        
        for well in wells:
            if well in new_state['_index']:
                owner = new_state['_index'][well]
                conflicts.append((well, owner))
            else:
                new_state['frames'][name][well] = {
                    'dilution': dilution,
                    'od': None
                }
                new_state['_index'][well] = name
                added.append(well)
    
    # Отчет
    report = f"📦 Фрейм '{name}':\n"
    if added:
        report += f"  ✅ Добавлено: {', '.join(added)}\n"
    if conflicts:
        report += f"  ⚠️ Пропущено (занято):\n"
        for well, owner in conflicts:
            report += f"     {well} → '{owner}'\n"
    
    return new_state, report

In [12]:
def execute_edit(state, name, wells, new_dilution):
    """Изменить разведение у конкретных лунок во фрейме"""
    new_state = deepcopy(state)
    
    if name not in new_state['frames']:
        raise ValueError(f"Фрейм '{name}' не существует")
    
    edited = []
    not_found = []
    
    for well in wells:
        if well in new_state['frames'][name]:
            new_state['frames'][name][well]['dilution'] = new_dilution
            edited.append(well)
        else:
            not_found.append(well)
    
    # Отчет
    report = f"✏️ Фрейм '{name}':\n"
    if edited:
        report += f"  ✅ Изменено: {', '.join(edited)} → dil {new_dilution}\n"
    if not_found:
        report += f"  ⚠️ Не найдены: {', '.join(not_found)}\n"
    
    return new_state, report

In [13]:
def execute_erase(state, name, wells):
    """Удалить лунки из фрейма"""
    new_state = deepcopy(state)
    
    if name not in new_state['frames']:
        raise ValueError(f"Фрейм '{name}' не существует")
    
    erased = []
    not_found = []
    
    for well in wells:
        if well in new_state['frames'][name]:
            del new_state['frames'][name][well]
            del new_state['_index'][well]
            erased.append(well)
        else:
            not_found.append(well)
    
    # Если фрейм стал пустым - удаляем его
    if name in new_state['frames'] and not new_state['frames'][name]:
        del new_state['frames'][name]
        report = f"🗑️ Фрейм '{name}' удален (стал пустым)\n"
    else:
        report = f"🧹 Фрейм '{name}':\n"
    
    if erased:
        report += f"  ✅ Удалено: {', '.join(erased)}\n"
    if not_found:
        report += f"  ⚠️ Не найдены: {', '.join(not_found)}\n"
    
    return new_state, report

In [14]:
def execute_rename(state, old_name, new_name):
    """Переименовать фрейм"""
    new_state = deepcopy(state)
    
    if old_name not in new_state['frames']:
        raise ValueError(f"Фрейм '{old_name}' не существует")
    
    if new_name in new_state['frames']:
        raise ValueError(f"Фрейм '{new_name}' уже существует")
    
    # Переименовываем фрейм
    new_state['frames'][new_name] = new_state['frames'].pop(old_name)
    
    # Обновляем индекс
    for well in new_state['frames'][new_name]:
        new_state['_index'][well] = new_name
    
    report = f"📝 Фрейм '{old_name}' → '{new_name}'"
    
    return new_state, report

In [15]:
def execute_delete(state, name):
    """Удалить фрейм целиком"""
    new_state = deepcopy(state)
    
    if name not in new_state['frames']:
        raise ValueError(f"Фрейм '{name}' не существует")
    
    # Собираем лунки для отчета
    wells = list(new_state['frames'][name].keys())
    
    # Удаляем из индекса
    for well in wells:
        del new_state['_index'][well]
    
    # Удаляем фрейм
    del new_state['frames'][name]
    
    report = f"🗑️ Фрейм '{name}' удален\n"
    if wells:
        report += f"   Освобождено: {', '.join(wells[:5])}"
        if len(wells) > 5:
            report += f" и еще {len(wells)-5}"
    
    return new_state, report

In [16]:
def execute_show(state, target, name=None):
    """Показать информацию"""
    if target == 'all':
        if not state['frames']:
            return state, "📭 Нет фреймов"
        
        report = "📋 ФРЕЙМЫ:\n"
        for frame_name, wells in state['frames'].items():
            dilutions = set(w['dilution'] for w in wells.values())
            report += f"  • '{frame_name}': {len(wells)} лунок, dil: {sorted(dilutions)}\n"
        
        free = 96 - len(state['_index'])
        report += f"\n🟢 Свободно лунок: {free}/96"
        
    elif target == 'frame':
        if name not in state['frames']:
            raise ValueError(f"Фрейм '{name}' не существует")
        
        wells = state['frames'][name]
        report = f"🔬 Фрейм '{name}':\n"
        report += "   Лунка  Dilution  OD\n"
        for well, data in sorted(wells.items()):
            od = f"{data['od']:.3f}" if data['od'] else '--'
            report += f"   {well:<6} {data['dilution']:<8} {od}\n"
    
    return state, report

In [17]:
def execute_command(state, command):
    """Дирижер исполнителей"""
    action = command['action']
    
    if action == 'add':
        return execute_add(state, command['name'], command['specs'])
    
    elif action == 'edit':
        return execute_edit(state, command['name'], command['wells'], command['new_dilution'])
    
    elif action == 'erase':
        return execute_erase(state, command['name'], command['wells'])
    
    elif action == 'rename':
        return execute_rename(state, command['old_name'], command['new_name'])
    
    elif action == 'delete':
        return execute_delete(state, command['name'])
    
    elif action == 'show':
        return execute_show(state, command['target'], command.get('name'))
    elif action == 'load_full':
        return execute_load_full(state, command['file_path'], command['sheet_name'])
    
    elif action == 'undo':
        return state, "Undo"
    
    
    else:
        raise ValueError(f"Неизвестное действие: {action}")

Работа с экселем

In [18]:
def load_full_plate(file_path, sheet_name):
    wb = load_workbook(file_path, data_only=True)
    ws = wb[sheet_name]
    
    # 1. Ищем "Результаты" в колонке A
    results_row = None
    for row in range(1, 150):
        if ws.cell(row=row, column=1).value == "Результаты":
            results_row = row
            break
    
    if results_row is None:
        raise ValueError("Не найдено слово 'Результаты' в колонке A")
    
    # 2. A1 планшета = строка results_row + 4, колонка 3 (C)
    a1_row = results_row + 4
    a1_col = 3  # C = 3
    
    plate_data = {}
    rows = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
    
    for i, row_letter in zip(range(0, 32, 5), rows): # целится по read 2:550 по каждой букве в планшете
        for col in range(12):  # 12 колонок планшета
            well = f"{row_letter}{col+1}"
            od_value = ws.cell(
                row=a1_row + i,           # A -> +0, B -> +1, C -> +2...
                column=a1_col + col       # 1я колонка -> +0, 2я -> +1...
            ).value
            plate_data[well] = od_value
    
    wb.close()
    return plate_data

Инициализация фрейма

In [19]:
def initial_state():
    """Создает пустой эксперимент"""
    return {
        'frames': {},  # имя фрейма -> {лунка: {dilution: N, od: None}}
        '_index': {}   # лунка -> имя фрейма
    }

REPL

In [21]:
# @title 🎮 Запуск эксперимента { display-mode: "form" }
# Состояние
state = initial_state()
history = [deepcopy(state)]

print("🔬 96-луночный планшет")
print("Команды: add, edit, erase, rename, delete, show, undo")
print('Пример: add "образец 1" dil 20 B1:B3, dil 10 C1:C3')
print("-" * 40)

while True:
    try:
        cmd = input('> ')
        
        if cmd.lower() == 'undo':
            if len(history) > 1:
                history.pop()
                state = deepcopy(history[-1])
                print("✅ Отменено")
            else:
                print("⚠️ Нечего отменять")
            continue
            
        if cmd.lower() == 'exit':
            print("👋 Пока")
            break
            
        if not cmd.strip():
            continue
        
        # Парсим и выполняем
        command = parse_command(cmd)
        state, report = execute_command(state, command)
        
        # Сохраняем в историю
        history.append(deepcopy(state))
        
        # Выводим отчет
        print(report)
        print()
        
    except Exception as e:
        print(f"❌ Ошибка: {e}")

🔬 96-луночный планшет
Команды: add, edit, erase, rename, delete, show, undo
Пример: add "образец 1" dil 20 B1:B3, dil 10 C1:C3
----------------------------------------
⚠️ Нечего отменять
👋 Пока
